# EUDR M3 Siamese-DeepLabV3 Training

Change detection model: shared ResNet50 encoder processes 2020 and 2024 Sentinel-2 composites
separately, then takes the absolute feature difference to detect post-EUDR forest loss.

**Labels:** Hansen GFC 2025 v1.13 — pixel value 2 = forest loss 2021-2024 (post-EUDR).

**Accelerator:** T4 x2 — enable via *Settings → Accelerator → GPU T4 x2*  
**⚠️ P100 is NOT supported** — current PyTorch requires sm_70+; P100 is sm_60. Use T4 only.  
**Expected runtime:** ~2h (20 epochs, ~1020 farms, batch=4 across 2 GPUs)

### Kaggle Datasets to attach:
- `eudr-satellite` — contains `2020_baseline/`, `2024_current/`, `hansen_labels/`, `farms_osm.csv`

### Kaggle Secrets required:
- `GEE_PROJECT_ID`
- `GITHUB_TOKEN` — personal access token with `repo` scope (needed to clone private repo)

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    cc = props.major * 10 + props.minor
    note = "OK" if cc >= 70 else f"INCOMPATIBLE (sm_{cc} < sm_70) — switch accelerator to T4"
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB | sm_{cc} [{note}]")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable T4 x2 via Settings → Accelerator.")

In [ ]:
# Clone repo — uses GITHUB_TOKEN secret for private repo access
import subprocess, os

BRANCH   = "main"
WORK_DIR = "/kaggle/working/eudr"

if not os.path.exists(WORK_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        repo_url = f"https://{token}@github.com/rajul-kk/EUDR_compliance.git"
    except Exception:
        repo_url = "https://github.com/rajul-kk/EUDR_compliance.git"
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, WORK_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import hashlib, pathlib, subprocess

_req = pathlib.Path("requirements.txt").read_bytes()
_req_hash = hashlib.md5(_req).hexdigest()
_marker = pathlib.Path("/kaggle/working/.pip_hash")

if not _marker.exists() or _marker.read_text() != _req_hash:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
    _marker.write_text(_req_hash)
else:
    print("pip cache hit — skipping install")

In [ ]:
import os

DATA = "/kaggle/input/datasets/rajulkabir/eudr-satellite"

T1_DIR     = f"{DATA}/2020_baseline"
T2_DIR     = f"{DATA}/2024_current"
LABEL_DIR  = f"{DATA}/hansen_labels"
FARMS_CSV  = f"{DATA}/farms_osm.csv"

for p in [T1_DIR, T2_DIR, LABEL_DIR, FARMS_CSV]:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"[{status}] {p}")

In [ ]:
import sys, os, logging
sys.path.insert(0, "/kaggle/working/eudr")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

from src.preprocessing.change_dataset import ChangeDetectionDataset
import torch

# Redirect C-level stderr to /dev/null during dataset build (suppresses GDAL TIFF warnings)
_devnull = os.open(os.devnull, os.O_WRONLY)
_old_stderr = os.dup(2)
os.dup2(_devnull, 2)
os.close(_devnull)
try:
    ds = ChangeDetectionDataset(
        t1_dir=T1_DIR,
        t2_dir=T2_DIR,
        mask_dir=LABEL_DIR,
        label_backend="hansen",
    )
finally:
    sys.stderr.flush()
    os.dup2(_old_stderr, 2)
    os.close(_old_stderr)

print(f"Valid pairs: {len(ds)}")

# Sample 10 items to estimate class balance
loss_px = no_change_px = 0
for i in range(min(10, len(ds))):
    _, _, cm = ds[i]
    valid = cm[cm != 255]
    loss_px      += (valid == 1).sum().item()
    no_change_px += (valid == 0).sum().item()

total = loss_px + no_change_px
print(f"Forest loss pixels (sample 10): {loss_px} ({100*loss_px/max(total,1):.2f}%)")
print(f"No-change pixels   (sample 10): {no_change_px} ({100*no_change_px/max(total,1):.2f}%)")

In [ ]:
import sys, os, argparse, logging
sys.path.insert(0, "/kaggle/working/eudr")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
os.makedirs("/kaggle/working/models", exist_ok=True)

from src.change_siamese_train import train

args = argparse.Namespace(
    t1_dir=T1_DIR, t2_dir=T2_DIR, mask_dir=LABEL_DIR,
    label_backend="hansen",
    output_model_path="/kaggle/working/models/farm_siamese.pth",
    epochs=20, batch_size=4, learning_rate=1e-4,
    val_ratio=0.15, patience=7, seed=42,
    warmup_epochs=3, focal_gamma=2.0, focal_alpha=0.97,
)

# Suppress C-level GDAL stderr; tqdm/logging write to stdout so they stay visible
_devnull = os.open(os.devnull, os.O_WRONLY)
_old_stderr = os.dup(2)
os.dup2(_devnull, 2)
os.close(_devnull)
try:
    train(args)
finally:
    sys.stderr.flush()
    os.dup2(_old_stderr, 2)
    os.close(_old_stderr)

In [ ]:
# Verify saved checkpoint
import torch
ckpt = torch.load("/kaggle/working/models/farm_siamese.pth", map_location="cpu")
print(f"Best epoch : {ckpt['best_epoch']}")
print(f"Best val F1: {ckpt['best_val_f1']:.4f}")
print(f"State dict keys (first 5): {list(ckpt['state_dict'].keys())[:5]}")

In [ ]:
# Package model for download
!zip /kaggle/working/farm_siamese.zip /kaggle/working/models/farm_siamese.pth
print("Download farm_siamese.zip from the Output panel.")